# Representações dos enunciados

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Igorbraziel/juristcu-clustering-tsdae/blob/main/notebooks/02_embeddings.ipynb)

- **Objetivo:** calcular as representações R1, R2, R2-CLS, R3 e R4 dos enunciados.
- **Entradas:** documentos do JurisTCU e corpus de frases salvo pelo `01_data_corpus`.
- **Saídas:** representações normalizadas (L2) e modelos do TSDAE no diretório de artefatos.
- **Requisitos:** GPU (T4). O TSDAE leva de 5 a 15 minutos por semente.
- Cada representação é salva logo após o cálculo. Se a sessão cair, as representações já salvas são preservadas.

In [ ]:
# Configuração do ambiente. Fora do Colab, usa o ambiente local criado por `uv sync`.
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"  # Trocar pelo nome de uma branch de trabalho para testá-la.
REPO_URL = "https://github.com/Igorbraziel/juristcu-clustering-tsdae.git"
REPO_DIR = "/content/juristcu-clustering-tsdae"


def run(command: str) -> None:
    """Executa um comando de shell e interrompe a célula em caso de erro."""
    subprocess.run(command, shell=True, check=True)


if IN_COLAB:
    # Clona o repositório, se necessário, e posiciona na versão remota da branch.
    run(f"test -d {REPO_DIR} || git clone -q {REPO_URL} {REPO_DIR}")
    run(f"git -C {REPO_DIR} fetch -q origin {BRANCH}")
    run(f"git -C {REPO_DIR} checkout -q -B {BRANCH} FETCH_HEAD")
    # O uv costuma vir no Colab. Instala apenas se faltar.
    run("command -v uv > /dev/null || pip install -q uv")
    # --system: instala no Python do Colab, que já traz o torch com CUDA.
    # --no-sources: ignora o índice de torch CPU do pyproject.toml.
    # --no-config: ignora configurações locais do uv.
    # Instalação não editável: disponível sem reiniciar o ambiente de execução.
    # Nunca usar `uv sync` nem requirements.txt no Colab.
    run(f"uv pip install --system --no-config --no-sources -q {REPO_DIR}")
    # Tabelas e figuras são gravadas em results/, a partir da raiz do repositório.
    os.chdir(REPO_DIR)
    # Após atualizar o repositório com mudanças em src/, reiniciar o ambiente de execução.
    from google.colab import drive

    drive.mount("/content/drive")

In [ ]:
import importlib
import json

# Atualiza o cache de importação para reconhecer o pacote recém-instalado.
importlib.invalidate_caches()

from juristcu import data, embeddings, storage, tsdae  # noqa: E402
from juristcu.config import (  # noqa: E402
    BERTIMBAU_ID,
    E5_ID,
    E5_PREFIX,
    LSA_DIM,
    SEED,
    TEXT_COLUMN,
    TSDAE_SEEDS,
    TsdaeConfig,
)

storage.save_environment("02")

## 1. Entradas

- Enunciados de todos os documentos, na ordem do arquivo.
- Corpus de frases salvo pelo `01_data_corpus`.

In [ ]:
documents = data.load_documents()
texts = documents[TEXT_COLUMN].tolist()
sentences = json.loads((storage.artifacts_dir() / "sentences.json").read_text(encoding="utf-8"))
len(texts), len(sentences)

## 2. R1: LSA

- TF-IDF com SVD truncado em 300 dimensões.
- Referência clássica, sem rede neural.

In [ ]:
r1 = embeddings.encode_lsa(texts, n_components=LSA_DIM, seed=SEED)
storage.save_embeddings("R1", embeddings.l2_normalize(r1))

## 3. R2: BERTimbau com pooling pela média

- BERTimbau base sem ajuste.
- Limite inferior neural.

In [ ]:
bertimbau_mean = embeddings.load_sentence_model(BERTIMBAU_ID, pooling="mean")
r2 = embeddings.encode_texts(bertimbau_mean, texts)
storage.save_embeddings("R2", embeddings.l2_normalize(r2))

## 4. R2-CLS: BERTimbau com pooling CLS

- Controle exploratório. Mesmo modelo de R2, com o pooling de R3.
- Isola o efeito do TSDAE na comparação com R3. Ver `docs/decisions.md`.

In [ ]:
bertimbau_cls = embeddings.load_sentence_model(BERTIMBAU_ID, pooling="cls")
r2_cls = embeddings.encode_texts(bertimbau_cls, texts)
storage.save_embeddings("R2-CLS", embeddings.l2_normalize(r2_cls))

## 5. R3: BERTimbau com TSDAE

- Um treinamento por semente, sempre com o mesmo corpus de frases.
- O modelo é carregado com o pooling salvo no treinamento (CLS).
- Avisos de pesos ausentes no decodificador são esperados. Ver `docs/decisions.md`.

In [ ]:
for seed in TSDAE_SEEDS:
    model_path = tsdae.train_tsdae(
        sentences,
        seed=seed,
        output_dir=storage.artifacts_dir() / "models" / f"tsdae-seed{seed}",
        config=TsdaeConfig(),
    )
    tsdae_model = embeddings.load_sentence_model(str(model_path))
    r3 = embeddings.encode_texts(tsdae_model, texts)
    storage.save_embeddings(f"R3-seed{seed}", embeddings.l2_normalize(r3))

## 6. R4: multilingual-e5-small

- Modelo geral pronto, com o pooling salvo pelos autores (média).
- Cada enunciado recebe o prefixo `query: `.

In [ ]:
e5 = embeddings.load_sentence_model(E5_ID)
r4 = embeddings.encode_texts(e5, texts, prefix=E5_PREFIX)
storage.save_embeddings("R4", embeddings.l2_normalize(r4))